In [4]:
from google.colab import userdata
import os
os.environ["LANGCHAIN_TRACING"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('langsmith')
os.environ["LANGCHAIN_PROJECT"] = "Personal Chef"

In [5]:
!pip install -U langchain-google-genai -q
from langchain_google_genai import ChatGoogleGenerativeAI

In [6]:
API_key = userdata.get('gemini')

llm_model = "gemini-2.5-flash"
os.environ["GOOGLE_API_KEY"] = API_key
model = ChatGoogleGenerativeAI(
    model=llm_model,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)


In [13]:
! pip install tavily -q
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily = TavilyClient(api_key=userdata.get('tavily'))

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily.search(query)

In [14]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [15]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [16]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

[{'type': 'text', 'text': 'With your leftover chicken and rice, you could easily make:\n\n1.  **Chicken Fried Rice**\n2.  **Chicken and Rice Enchilada Casserole**\n\nWould you like the instructions for either of these, or perhaps some other suggestions?', 'extras': {'signature': 'CtkDAb4+9vtwZp6Z3u/A1z+/ccqHbIZ6ny8gTeX1ZW8sHq81yOTAafzJfvhzcUSLNaUCU3qiaj4uR5z4U57xG2VfyBYYrWQ/lAiHcpUAQIq1eTpZz2t1gRr2CwBBh4j9XiEWcoos9Mn84Jv0cYlvC5VphnhduobtO3XuTa/meclWSaxQ/ibYmkfcvnwyH8rPlMt1Mqh2RHlQb/w+ScWM7cmTP3eFdoBSVvuVay5u3PfwcHEOx5cICL2i7jEUTkaAltCKQ2gLJYeKR+ANJLcEGHbkx4vgXu6uyYXo/ou0r+0Y0yrwjszjHmntcsxAtaiWiLn/VzyZwOptNxBwdLXGCZ+0Bw+npQUrJuo7WTgZxkH9hN0kSbVMlDyBx68xx2M2uqClEQUtOAFFTnzKuUmOmlSysbGW+L1fZA9wX/X+mBXziiD0ehaiPzwH7A1QvrLGDRoazidUwIo9Z6Be0xGxEXmOUb4ftPp6xO8YwKeF8sZ6hjjNo06gKFeyiHJKXyFGmeAD9vahXr5CmA6Wwi8e8LcMpTTUSyn/i0bCFuJhNoa63IZqCRoU1q3w1V/LbnVwTktZICfXK/GmfQ1fj6CGYBcrfi8Wg7HpGjtut2hj+ww1VrO4I9XjJ2TrjEo='}}]


In [17]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover chicken and rice. What can I make?', additional_kwargs={}, response_metadata={}, id='f30b9e3e-a62a-4394-ad0b-ebd03972ce60'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "recipes with leftover chicken and rice"}'}, '__gemini_function_call_thought_signatures__': {'48aaaf8b-f3b2-459d-b71c-36d5f6cda5a8': 'CsABAb4+9vvgH7YEGkxG7A17pU44nuRC90UKBriU3Vd2RbOpGjt3/iJTXowRTB5isSVME+bJeie8/XUDBiagSo3Xhia91E1Ty7Tzrv+asE4AYjOlZod1ZaK0nMHOOios1AShVlceaPFd0LHdb2gotzA9shxzw8DozreBnLX391uInOP5sle6aho3keeNFpLD6LMfxDyNcm8dG007Bue0iP3hmfFXELMZQGcuZzwlr/iTItA3NUNsGrAk5Hb1qzT2fpXR'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cbe5d-d8c4-71c0-9a3c-711fec870887-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'recipes with leftover chicken and rice'}, 'id': '48aaaf8b-